# 03 — Feature Engineering, Model Training & Tuning (Lending Club)

Core modeling notebook: turn the cleaned dataset into leakage-safe model matrices, train
and compare model families, run controlled feature-ablation experiments, tune LightGBM
with a strictly temporal validation scheme, and produce the champion models logged to
MLflow.

**Pipeline in this notebook**

1. Encode application-time attributes with fixed business-defined mappings; derive credit-history length; one-hot fit on the train window only.
2. Temporal split: train = 2012–2015 vintages, test = 2016–2018 (out-of-time).
3. Baselines: logistic regression vs LightGBM with comparable fixed hyperparameters.
4. Feature ablations: behavior-only, fully independent, enriched ratios, no-grade/no-rate.
5. Hyperparameter tuning: 25-config random search validated on a 2015 train-tail split — the test set is untouched until the final one-pass evaluation.
6. Final tuned model refit and single evaluation on the out-of-time window.

Every training run is logged to the MLflow experiment
`/Users/<your_username>/credit_risk_lending_club` with a variant tag, so all results
are reproducible from tracked artifacts. Run the `%pip install lightgbm` cell first in
a fresh session.

In [1]:
# LightGBM is not preinstalled on this compute — install it (notebook-scoped).
# This cell must be re-run in each new session before the LightGBM training cell.

%pip install lightgbm

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import pandas as pd

# Load the cleaned dataset from Unity Catalog
df_lending_club = spark.read.table('workspace.default.lending_club_cleaned').toPandas()


In [3]:
#Check the data

df_lending_club.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,acc_now_delinq,pub_rec_bankruptcies,tax_liens,mort_acc,tot_cur_bal,tot_coll_amt,collections_12_mths_ex_med,emp_length_missing,has_delinq_history,has_public_record,tot_cur_bal_missing,tot_coll_amt_missing,loan_default
0,10000.0,10000.0,10000.0,36 months,9.76,321.55,B,B1,10,MORTGAGE,180468.0,Verified,Jun-2012,Fully Paid,debt_consolidation,CA,22.14,0.0,Jul-1977,725.0,729.0,1.0,31.0,74.0,18.0,0.0,37770.0,69.0,40.0,f,Individual,0.0,0.0,0.0,4.0,79234.0,0.0,0.0,0,0,0,1,1,0
1,10000.0,10000.0,10000.0,36 months,14.65,344.95,C,C2,6,RENT,50000.0,Source Verified,Jun-2012,Charged Off,credit_card,CO,25.21,1.0,Jul-2003,670.0,674.0,0.0,23.0,74.0,14.0,0.0,8541.0,69.0,20.0,f,Individual,0.0,0.0,0.0,0.0,79234.0,0.0,0.0,0,1,0,1,1,1
2,12000.0,12000.0,12000.0,36 months,10.74,391.39,B,B2,3,MORTGAGE,70000.0,Verified,Jul-2012,Charged Off,debt_consolidation,MD,16.83,0.0,Mar-1996,700.0,704.0,0.0,66.0,74.0,12.0,0.0,9222.0,42.5,25.0,f,Individual,0.0,0.0,0.0,2.0,79234.0,0.0,0.0,0,1,0,1,1,1
3,20000.0,20000.0,20000.0,36 months,19.99,743.17,E,E1,-1,RENT,45000.0,Source Verified,Jun-2012,Fully Paid,credit_card,MA,21.63,0.0,May-2001,660.0,664.0,1.0,73.0,74.0,11.0,0.0,21466.0,64.9,30.0,f,Individual,0.0,0.0,0.0,0.0,79234.0,0.0,0.0,1,1,0,1,1,0
4,5600.0,5600.0,5600.0,36 months,14.65,193.17,C,C2,7,MORTGAGE,150000.0,Not Verified,Jun-2012,Fully Paid,medical,SC,10.70,0.0,Aug-2001,670.0,674.0,3.0,66.0,74.0,12.0,0.0,5676.0,51.7,27.0,f,Individual,0.0,0.0,0.0,5.0,79234.0,0.0,0.0,0,1,0,1,1,0


## 1. Feature encoding

All encodings use fixed, business-defined mappings — nothing is learned from the data,
so there is no leakage risk. Ordinal `grade`/`sub_grade` encode LC’s own risk ranking
(A1 best → G5 worst). `credit_history_months` measures the borrower’s credit tenure at
application; rare categories are collapsed so one-hot encoding stays clean.

In [4]:
# Encode simple categorical columns with known, business-defined mappings:
#   term              -> loan length in months (36 / 60); chosen at application, NOT leakage
#   initial_list_status -> 'w' = whole loan (single investor) / 'f' = fractional -> 1 / 0
#   application_type  -> Individual -> 0, Joint App -> 1
#   grade / sub_grade -> LC's own risk grades, ordinal by risk (A1 best ... G5 worst)

df_lending_club['term'] = df_lending_club['term'].str.strip().str.replace(' months', '').astype(int)
df_lending_club['initial_list_status'] = df_lending_club['initial_list_status'].map({'w': 1, 'f': 0})
df_lending_club['application_type'] = df_lending_club['application_type'].map({'Individual': 0, 'Joint App': 1})

df_lending_club['grade'] = df_lending_club['grade'].map({g: i for i, g in enumerate('ABCDEFG')})
df_lending_club['sub_grade'] = [
    (ord(s[0]) - 65) * 5 + int(s[1]) - 1 for s in df_lending_club['sub_grade']
]

print(df_lending_club[['term', 'initial_list_status', 'application_type', 'grade', 'sub_grade']].head())

   term  initial_list_status  application_type  grade  sub_grade
0    36                    0                 0      1          5
1    36                    0                 0      2         11
2    36                    0                 0      1          6
3    36                    0                 0      4         20
4    36                    0                 0      2         11


In [5]:
# Credit history length: how long the borrower has been using credit, in months,
# measured from their earliest credit line to the loan issue date (both known at application).
# The raw strings like 'Aug-2003' are parsed with the '%b-%Y' format.

cr_line = pd.to_datetime(df_lending_club['earliest_cr_line'], format='%b-%Y')
issue = pd.to_datetime(df_lending_club['issue_d'], format='%b-%Y')

df_lending_club['credit_history_months'] = (
    (issue.dt.year - cr_line.dt.year) * 12 + (issue.dt.month - cr_line.dt.month)
)
# a handful of records have credit lines dated after the issue date (data errors) -> floor at 0
df_lending_club['credit_history_months'] = df_lending_club['credit_history_months'].clip(lower=0)

print(df_lending_club['credit_history_months'].describe())

count    1.296898e+06
mean     1.961678e+02
std      9.019395e+01
min      3.600000e+01
25%      1.350000e+02
50%      1.780000e+02
75%      2.420000e+02
max      9.990000e+02
Name: credit_history_months, dtype: float64


In [6]:
# Collapse rare categories so one-hot encoding stays clean:
#   home_ownership: ANY (285) + NONE (45) + OTHER (45) -> 'OTHER'
#   purpose: 'educational' (1 row) -> 'other'

df_lending_club['home_ownership'] = df_lending_club['home_ownership'].replace({'ANY': 'OTHER', 'NONE': 'OTHER'})
df_lending_club['purpose'] = df_lending_club['purpose'].replace({'educational': 'other'})

print(df_lending_club['home_ownership'].value_counts())
print()
print(df_lending_club['purpose'].value_counts())

home_ownership
MORTGAGE    644119
RENT        511374
OWN         141030
OTHER          375
Name: count, dtype: int64

purpose
debt_consolidation    757087
credit_card           288480
home_improvement       84014
other                  73252
major_purchase         26994
medical                14743
small_business         13217
car                    12814
moving                  8827
vacation                8626
house                   6779
wedding                 1249
renewable_energy         816
Name: count, dtype: int64


In [7]:
# Drop columns that should not be model features:
#   loan_status       -> source of the target (keeping it would be direct leakage)
#   funded_amnt(_inv) -> duplicates of loan_amnt, no added signal
#   addr_state        -> 51 levels, marginal signal for a baseline model
#   earliest_cr_line  -> replaced by credit_history_months
#   issue_d           -> kept only until the temporal split, then dropped there

drop_cols = ['loan_status', 'funded_amnt', 'funded_amnt_inv', 'addr_state', 'earliest_cr_line']
df_lending_club = df_lending_club.drop(columns=drop_cols)

print(f'remaining columns ({len(df_lending_club.columns)}):')
print(list(df_lending_club.columns))

remaining columns (40):
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'pub_rec_bankruptcies', 'tax_liens', 'mort_acc', 'tot_cur_bal', 'tot_coll_amt', 'collections_12_mths_ex_med', 'emp_length_missing', 'has_delinq_history', 'has_public_record', 'tot_cur_bal_missing', 'tot_coll_amt_missing', 'loan_default', 'credit_history_months']


## 2. Temporal split and model matrices

Train on 2012–2015 vintages, test on 2016–2018 — an out-of-time evaluation that mirrors
production (learn from the past, score the future). The 2016-01-01 cutoff also gives
every test loan at least two years of seasoning, limiting survivorship bias in the
most recent vintages. One-hot encodings are fit on the train window only; the test
matrix is reindexed onto the train columns so no test category leaks into training.

In [8]:
# Temporal split: train on loans issued 2012-2015, test on 2016-2018. This mirrors
# production use — scoring newer applications with patterns learned from the past —
# and stops the test set from borrowing information from the future. No shuffling,
# no stratifying across time.
#
# CUTOFF NOTE: the data ends Dec-2018 and a loan can only appear 'resolved' once it has
# either fully repaid (3-5 years) or charged off (1-2 years). Very recent vintages are
# therefore biased: they only show up if they prepaid almost immediately or defaulted
# almost immediately (H2-2018 measured 7.5% default vs ~20% overall). The 2016-01-01
# cutoff gives every test loan at least 2 years of seasoning, keeping that bias small.

issue_month = pd.to_datetime(df_lending_club['issue_d'], format='%b-%Y')
cutoff = pd.Timestamp('2016-01-01')

train_df = df_lending_club[issue_month < cutoff].reset_index(drop=True)
test_df = df_lending_club[issue_month >= cutoff].reset_index(drop=True)

# issue_d has served its purpose (defining the split); drop it from both frames
train_df = train_df.drop(columns='issue_d')
test_df = test_df.drop(columns='issue_d')

print(f'train: {len(train_df):,} rows, default rate {train_df["loan_default"].mean():.1%}')
print(f'test:  {len(test_df):,} rows, default rate {test_df["loan_default"].mean():.1%}')

train: 778,940 rows, default rate 18.7%
test:  517,958 rows, default rate 22.4%


In [9]:
# One-hot encode the remaining categoricals, FIT ON TRAIN ONLY: dummies are built from
# the training data and the test set is reindexed onto the same columns, so no test-set
# categories leak into the training matrix. drop_first=True avoids collinearity for LR.

onehot_cols = ['home_ownership', 'verification_status', 'purpose']

train_encoded = pd.get_dummies(train_df, columns=onehot_cols, drop_first=True, dtype=int)
test_encoded = pd.get_dummies(test_df, columns=onehot_cols, drop_first=True, dtype=int).reindex(
    columns=train_encoded.columns, fill_value=0
)

# separate features and target -> model-ready matrices
y_train = train_encoded.pop('loan_default')
y_test = test_encoded.pop('loan_default')
X_train = train_encoded
X_test = test_encoded

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'default rate — train: {y_train.mean():.1%}, test: {y_test.mean():.1%}')

X_train: (778940, 52), X_test: (517958, 52)
default rate — train: 18.7%, test: 22.4%


In [10]:
# Sanity check the model-ready matrices: shapes align, no nulls, everything numeric

assert X_train.shape[1] == X_test.shape[1], 'train/test column mismatch'
assert X_train.shape[0] == y_train.shape[0] and X_test.shape[0] == y_test.shape[0], 'row/target mismatch'

print(f'nulls — X_train: {X_train.isnull().sum().sum()}, X_test: {X_test.isnull().sum().sum()}')
print(f'non-numeric columns remaining: {X_train.select_dtypes(exclude="number").columns.tolist()}')
print()
print(X_train.dtypes.value_counts())
print()
print(X_train.columns.tolist())

nulls — X_train: 0, X_test: 0
non-numeric columns remaining: []

int64      28
float64    23
int32       1
Name: count, dtype: int64

['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'acc_now_delinq', 'pub_rec_bankruptcies', 'tax_liens', 'mort_acc', 'tot_cur_bal', 'tot_coll_amt', 'collections_12_mths_ex_med', 'emp_length_missing', 'has_delinq_history', 'has_public_record', 'tot_cur_bal_missing', 'tot_coll_amt_missing', 'credit_history_months', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'verification_status_Source Verified', 'verification_status_Verified', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'pu

## 3. Baseline models

Logistic regression (interpretable baseline) vs LightGBM (non-linear challenger), both
with `class_weight='balanced'` and comparable fixed hyperparameters, both logged to
MLflow. Balanced class weights inflate raw probabilities — scores are used as rankings
here (AUC / KS / deciles); PD calibration happens in the business-impact notebook.

In [11]:
# Train the logistic regression baseline with MLflow experiment tracking.
# class_weight='balanced' compensates for defaults being the minority class; note this
# inflates the absolute predicted probabilities, so treat scores as RANKINGS here
# (AUC / KS / deciles) and recalibrate before using them as raw PDs.

import time
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

start = time.time()
with mlflow.start_run(run_name='logistic_regression_baseline') as run:
    pipeline.fit(X_train, y_train)

    prob_train = pipeline.predict_proba(X_train)[:, 1]
    prob_test = pipeline.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, prob_train)
    test_auc = roc_auc_score(y_test, prob_test)
    test_logloss = log_loss(y_test, prob_test)

    mlflow.log_params({
        'model': 'logistic_regression', 'max_iter': 1000,
        'class_weight': 'balanced', 'features_scaled': True,
        'n_features': X_train.shape[1], 'n_train': X_train.shape[0],
        'n_test': X_test.shape[0], 'split': 'temporal_2016_01_01',
    })
    mlflow.log_metrics({
        'train_auc': train_auc, 'test_auc': test_auc, 'test_log_loss': test_logloss,
    })

    signature = infer_signature(X_train.head(100), pipeline.predict_proba(X_train.head(100))[:, 1])
    mlflow.sklearn.log_model(pipeline, name='model', signature=signature, input_example=X_train.head(3))
    print(f'MLflow run: {run.info.run_id}')

print(f'trained in {time.time() - start:.0f}s')
print(f'AUC — train: {train_auc:.3f}, test: {test_auc:.3f}')
print(f'test log-loss: {test_logloss:.3f}')

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-8aaea8a6e41b44efb3eb004dd257013f?o=7474655510040746


MLflow run: 0ee58eabc2b94fe288f3189910836a26
trained in 14s
AUC — train: 0.718, test: 0.707
test log-loss: 0.603


In [12]:
# Evaluate the baseline on the untouched test set: ranking power (AUC / KS / deciles),
# classification metrics at the default threshold, and calibration by decile.

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

prob_test = pipeline.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, prob_test)
pr_auc = average_precision_score(y_test, prob_test)
fpr, tpr, _ = roc_curve(y_test, prob_test)
ks = np.max(tpr - fpr)

print(f'ROC-AUC: {roc_auc:.3f}  (Gini: {2 * roc_auc - 1:.3f})')
print(f'PR-AUC:  {pr_auc:.3f}')
print(f'KS:      {ks:.3f}')
print(f'Brier:   {brier_score_loss(y_test, prob_test):.3f}')
print()
print(f"at threshold 0.5 — accuracy: {accuracy_score(y_test, pred_test):.3f}, "
      f"precision: {precision_score(y_test, pred_test):.3f}, "
      f"recall: {recall_score(y_test, pred_test):.3f}, F1: {f1_score(y_test, pred_test):.3f}")
print()
print('confusion matrix (rows = actual, cols = predicted):')
print(confusion_matrix(y_test, pred_test))

# Decile table: rank loans by predicted risk, show the actual default rate in each bucket.
# A good model concentrates defaults in the top deciles (high lift).
eval_df = pd.DataFrame({'prob': prob_test, 'actual': y_test.values})
eval_df['decile'] = pd.qcut(eval_df['prob'].rank(method='first'), 10,
                            labels=[f'D{i}' for i in range(1, 11)])
decile_table = eval_df.groupby('decile', observed=True).agg(
    n=('actual', 'count'),
    avg_predicted=('prob', 'mean'),
    actual_rate=('actual', 'mean'),
)
decile_table['lift'] = decile_table['actual_rate'] / eval_df['actual'].mean()
print()
print(decile_table.round(3))

ROC-AUC: 0.707  (Gini: 0.414)
PR-AUC:  0.397
KS:      0.302
Brier:   0.207

at threshold 0.5 — accuracy: 0.682, precision: 0.367, recall: 0.577, F1: 0.448

confusion matrix (rows = actual, cols = predicted):
[[286149 115669]
 [ 49173  66967]]

            n  avg_predicted  actual_rate   lift
decile                                          
D1      51796          0.168        0.047  0.211
D2      51796          0.257        0.087  0.386
D3      51796          0.317        0.123  0.549
D4      51795          0.367        0.156  0.697
D5      51796          0.411        0.189  0.843
D6      51796          0.455        0.226  1.008
D7      51795          0.501        0.265  1.184
D8      51796          0.556        0.307  1.369
D9      51796          0.628        0.360  1.607
D10     51796          0.757        0.481  2.147


In [13]:
# Train the LightGBM challenger: captures the non-linearities and interactions a
# linear model cannot. Fixed hyperparameters for a fair first comparison against the
# LR baseline; same MLflow experiment so the runs are directly comparable.
# class_weight='balanced' since defaults are the minority class (same calibration
# caveat as the baseline — treat scores as rankings, recalibrate for raw PDs).

import time
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')

lgbm_model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

start = time.time()
with mlflow.start_run(run_name='lightgbm_challenger') as run:
    lgbm_model.fit(X_train, y_train)

    prob_train = lgbm_model.predict_proba(X_train)[:, 1]
    prob_test = lgbm_model.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, prob_train)
    test_auc = roc_auc_score(y_test, prob_test)
    test_logloss = log_loss(y_test, prob_test)

    mlflow.log_params({
        'model': 'lightgbm', 'n_estimators': 500, 'learning_rate': 0.05,
        'num_leaves': 63, 'min_child_samples': 100, 'class_weight': 'balanced',
        'n_features': X_train.shape[1], 'n_train': X_train.shape[0],
        'n_test': X_test.shape[0], 'split': 'temporal_2016_01_01',
    })
    mlflow.log_metrics({
        'train_auc': train_auc, 'test_auc': test_auc, 'test_log_loss': test_logloss,
    })

    signature = infer_signature(X_train.head(100), lgbm_model.predict_proba(X_train.head(100))[:, 1])
    mlflow.sklearn.log_model(lgbm_model, name='model', signature=signature, input_example=X_train.head(3))
    print(f'MLflow run: {run.info.run_id}')

print(f'trained in {time.time() - start:.0f}s')
print(f'AUC — train: {train_auc:.3f}, test: {test_auc:.3f}')
print(f'test log-loss: {test_logloss:.3f}')

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-18518ff2b8ce4e4ba77bb3f4f9b108e2?o=7474655510040746


MLflow run: 018dcac1927c4206a58c9d0d45ad4330
trained in 77s
AUC — train: 0.758, test: 0.714
test log-loss: 0.592


In [14]:
# Evaluate the LightGBM challenger on the untouched test set — same metrics as the
# LR baseline for a direct comparison — plus feature importances (split gain share).

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

prob_test = lgbm_model.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, prob_test)
pr_auc = average_precision_score(y_test, prob_test)
fpr, tpr, _ = roc_curve(y_test, prob_test)
ks = np.max(tpr - fpr)

print(f'ROC-AUC: {roc_auc:.3f}  (Gini: {2 * roc_auc - 1:.3f})')
print(f'PR-AUC:  {pr_auc:.3f}')
print(f'KS:      {ks:.3f}')
print(f'Brier:   {brier_score_loss(y_test, prob_test):.3f}')
print()
print(f"at threshold 0.5 — accuracy: {accuracy_score(y_test, pred_test):.3f}, "
      f"precision: {precision_score(y_test, pred_test):.3f}, "
      f"recall: {recall_score(y_test, pred_test):.3f}, F1: {f1_score(y_test, pred_test):.3f}")
print()
print('confusion matrix (rows = actual, cols = predicted):')
print(confusion_matrix(y_test, pred_test))

# Decile table — same view as the baseline for a like-for-like ranking check
eval_df = pd.DataFrame({'prob': prob_test, 'actual': y_test.values})
eval_df['decile'] = pd.qcut(eval_df['prob'].rank(method='first'), 10,
                            labels=[f'D{i}' for i in range(1, 11)])
decile_table = eval_df.groupby('decile', observed=True).agg(
    n=('actual', 'count'),
    avg_predicted=('prob', 'mean'),
    actual_rate=('actual', 'mean'),
)
decile_table['lift'] = decile_table['actual_rate'] / eval_df['actual'].mean()
print()
print(decile_table.round(3))

# Feature importance: share of total split gain
gain = pd.Series(
    lgbm_model.booster_.feature_importance(importance_type='gain'),
    index=X_train.columns,
).sort_values(ascending=False)
print()
print('top 15 features by split-gain share:')
print((gain / gain.sum()).head(15).round(3))

ROC-AUC: 0.714  (Gini: 0.428)
PR-AUC:  0.406
KS:      0.312
Brier:   0.204

at threshold 0.5 — accuracy: 0.675, precision: 0.366, recall: 0.610, F1: 0.457

confusion matrix (rows = actual, cols = predicted):
[[278969 122849]
 [ 45319  70821]]

            n  avg_predicted  actual_rate   lift
decile                                          
D1      51796          0.109        0.041  0.183
D2      51796          0.214        0.084  0.373
D3      51796          0.290        0.119  0.533
D4      51795          0.353        0.153  0.683
D5      51796          0.408        0.188  0.838
D6      51796          0.461        0.226  1.006
D7      51795          0.513        0.264  1.178
D8      51796          0.568        0.309  1.379
D9      51796          0.638        0.369  1.644
D10     51796          0.771        0.490  2.184

top 15 features by split-gain share:
sub_grade                0.337
grade                    0.185
term                     0.088
dti                      0.043
annual

## 4. Feature-ablation experiments

Controlled experiments to find where the signal lives. Each variant drops a family of
LC-produced features and retrains with the same fixed settings; every run is tagged in
MLflow for side-by-side comparison.

* **behavior-only** — drop `grade` / `sub_grade` (LC’s grading), keep pricing
* **fully independent** — also drop `int_rate` / `installment` (LC’s pricing): raw borrower behavior only
* **enriched ratios** — add affordability ratios (loan/payment/revol-to-income, mortgage share)
* **no-grade / no-rate** — drop grading and rate, keep `installment` (pricing partially leaks back through the amortization formula)

In [15]:
# Behavior-only experiment: strip Lending Club's own risk grades (grade, sub_grade)
# and test how well default is predictable from raw borrower attributes alone.
# NOTE: int_rate stays for now — it is LC's own pricing and correlates strongly with
# sub_grade, so this variant still carries part of the grading signal (next step if
# the result is interesting: drop it too for a fully independent feature set).

X_train_bh = X_train.drop(columns=['grade', 'sub_grade'])
X_test_bh = X_test.drop(columns=['grade', 'sub_grade'])

print(f'features: {X_train.shape[1]} -> {X_train_bh.shape[1]}')
print('dropped:', sorted(set(X_train.columns) - set(X_train_bh.columns)))

features: 52 -> 50
dropped: ['grade', 'sub_grade']


In [16]:
# Train both model families on the behavior-only feature set with the same fixed
# settings as the graded versions; MLflow runs get a 'behavior_only' variant tag so
# all four runs can be compared side by side in the experiment.

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')
results = {}

# --- LightGBM: same fixed hyperparameters as the graded challenger ---
lgbm_bh = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=100, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1,
)

with mlflow.start_run(run_name='lightgbm_behavior_only') as run:
    lgbm_bh.fit(X_train_bh, y_train)
    auc_tr = roc_auc_score(y_train, lgbm_bh.predict_proba(X_train_bh)[:, 1])
    prob_test = lgbm_bh.predict_proba(X_test_bh)[:, 1]
    auc_te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'behavior_only')
    mlflow.log_params({'model': 'lightgbm', 'n_estimators': 500, 'learning_rate': 0.05,
                        'num_leaves': 63, 'n_features': X_train_bh.shape[1],
                        'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': auc_tr, 'test_auc': auc_te, 'test_log_loss': ll})
    signature = infer_signature(X_train_bh.head(100), lgbm_bh.predict_proba(X_train_bh.head(100))[:, 1])
    mlflow.sklearn.log_model(lgbm_bh, name='model', signature=signature, input_example=X_train_bh.head(3))
    print(f'lightgbm_behavior_only run: {run.info.run_id}')
results['lightgbm_behavior_only'] = (auc_tr, auc_te, ll)

# --- Logistic regression: same setup as the graded baseline ---
lr_bh = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

with mlflow.start_run(run_name='lr_behavior_only') as run:
    lr_bh.fit(X_train_bh, y_train)
    auc_tr = roc_auc_score(y_train, lr_bh.predict_proba(X_train_bh)[:, 1])
    prob_test = lr_bh.predict_proba(X_test_bh)[:, 1]
    auc_te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'behavior_only')
    mlflow.log_params({'model': 'logistic_regression', 'max_iter': 1000,
                        'n_features': X_train_bh.shape[1], 'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': auc_tr, 'test_auc': auc_te, 'test_log_loss': ll})
    signature = infer_signature(X_train_bh.head(100), lr_bh.predict_proba(X_train_bh.head(100))[:, 1])
    mlflow.sklearn.log_model(lr_bh, name='model', signature=signature, input_example=X_train_bh.head(3))
    print(f'lr_behavior_only run: {run.info.run_id}')
results['lr_behavior_only'] = (auc_tr, auc_te, ll)

print()
print('variant                      train_auc  test_auc  log_loss')
print(f"lightgbm graded (earlier)      0.756      0.713     0.594")
for name, (tr, te, ll) in results.items():
    print(f'{name:<30} {tr:.3f}      {te:.3f}     {ll:.3f}')
print(f"logreg graded (earlier)        0.718      0.706     0.606")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-77dbf851444f4bf7be5b7ff6a0e149c0?o=7474655510040746


lightgbm_behavior_only run: e88722c870ad4389b98037bc77817eff


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-0e57c94bc1744dd4ae51ef70c676aa26?o=7474655510040746


lr_behavior_only run: 60a6bb3950ce450daab5a2da4af9de20

variant                      train_auc  test_auc  log_loss
lightgbm graded (earlier)      0.756      0.713     0.594
lightgbm_behavior_only         0.756      0.713     0.605
lr_behavior_only               0.715      0.706     0.630
logreg graded (earlier)        0.718      0.706     0.606


In [17]:
# Evaluate the behavior-only LightGBM: decile separation and — the interesting
# part — which raw borrower attributes carry the signal once LC's grading is gone.

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

prob_bh = lgbm_bh.predict_proba(X_test_bh)[:, 1]

eval_df = pd.DataFrame({'prob': prob_bh, 'actual': y_test.values})
eval_df['decile'] = pd.qcut(eval_df['prob'].rank(method='first'), 10,
                            labels=[f'D{i}' for i in range(1, 11)])
decile_table = eval_df.groupby('decile', observed=True).agg(
    n=('actual', 'count'),
    avg_predicted=('prob', 'mean'),
    actual_rate=('actual', 'mean'),
)
decile_table['lift'] = decile_table['actual_rate'] / eval_df['actual'].mean()
print(decile_table.round(3))

gain = pd.Series(
    lgbm_bh.booster_.feature_importance(importance_type='gain'),
    index=X_train_bh.columns,
).sort_values(ascending=False)
print()
print('top 15 behavior features by split-gain share:')
print((gain / gain.sum()).head(15).round(3))

            n  avg_predicted  actual_rate   lift
decile                                          
D1      51796          0.113        0.042  0.185
D2      51796          0.215        0.083  0.370
D3      51796          0.288        0.121  0.540
D4      51795          0.349        0.154  0.688
D5      51796          0.407        0.190  0.847
D6      51796          0.464        0.222  0.990
D7      51795          0.525        0.266  1.185
D8      51796          0.592        0.307  1.369
D9      51796          0.673        0.371  1.654
D10     51796          0.798        0.487  2.170

top 15 behavior features by split-gain share:
int_rate                 0.499
term                     0.107
dti                      0.049
fico_range_low           0.034
tot_cur_bal              0.033
annual_inc               0.033
revol_bal                0.027
credit_history_months    0.024
mort_acc                 0.021
installment              0.020
emp_length               0.020
loan_amnt               

In [18]:
# Fully independent variant: on top of grade/sub_grade, also remove LC's PRICING —
# int_rate (set by LC from the sub-grade; it absorbed the grading signal in the
# behavior-only run) and installment (mechanically derived from amount/term/rate).
# What remains is raw borrower behavior: DTI, income, FICO, utilization, inquiries,
# delinquency history, credit length, plus loan size and term.

X_train_fi = X_train_bh.drop(columns=['int_rate', 'installment'])
X_test_fi = X_test_bh.drop(columns=['int_rate', 'installment'])

print(f'features: {X_train.shape[1]} (graded) -> {X_train_bh.shape[1]} (behavior) '
      f'-> {X_train_fi.shape[1]} (fully independent)')

features: 52 (graded) -> 50 (behavior) -> 48 (fully independent)


In [19]:
# Train both families on the fully independent feature set (no LC grading or pricing
# information at all) — the honest test of whether raw borrower behavior alone can
# predict default. Same fixed settings; MLflow tag 'fully_independent'.

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')
results_fi = {}

# --- LightGBM ---
lgbm_fi = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=100, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1,
)

with mlflow.start_run(run_name='lightgbm_fully_independent') as run:
    lgbm_fi.fit(X_train_fi, y_train)
    auc_tr = roc_auc_score(y_train, lgbm_fi.predict_proba(X_train_fi)[:, 1])
    prob_test = lgbm_fi.predict_proba(X_test_fi)[:, 1]
    auc_te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'fully_independent')
    mlflow.log_params({'model': 'lightgbm', 'n_estimators': 500, 'learning_rate': 0.05,
                        'num_leaves': 63, 'n_features': X_train_fi.shape[1],
                        'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': auc_tr, 'test_auc': auc_te, 'test_log_loss': ll})
    signature = infer_signature(X_train_fi.head(100), lgbm_fi.predict_proba(X_train_fi.head(100))[:, 1])
    mlflow.sklearn.log_model(lgbm_fi, name='model', signature=signature, input_example=X_train_fi.head(3))
    print(f'lightgbm_fully_independent run: {run.info.run_id}')
results_fi['lightgbm_fully_independent'] = (auc_tr, auc_te, ll)

# --- Logistic regression ---
lr_fi = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

with mlflow.start_run(run_name='lr_fully_independent') as run:
    lr_fi.fit(X_train_fi, y_train)
    auc_tr = roc_auc_score(y_train, lr_fi.predict_proba(X_train_fi)[:, 1])
    prob_test = lr_fi.predict_proba(X_test_fi)[:, 1]
    auc_te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'fully_independent')
    mlflow.log_params({'model': 'logistic_regression', 'max_iter': 1000,
                        'n_features': X_train_fi.shape[1], 'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': auc_tr, 'test_auc': auc_te, 'test_log_loss': ll})
    signature = infer_signature(X_train_fi.head(100), lr_fi.predict_proba(X_train_fi.head(100))[:, 1])
    mlflow.sklearn.log_model(lr_fi, name='model', signature=signature, input_example=X_train_fi.head(3))
    print(f'lr_fully_independent run: {run.info.run_id}')
results_fi['lr_fully_independent'] = (auc_tr, auc_te, ll)

print()
print('variant                        train_auc  test_auc  log_loss')
for name, tr, te, l in [
    ('lightgbm graded (earlier)',      0.756, 0.713, 0.594),
    ('lightgbm behavior-only (earlier)', 0.755, 0.712, 0.607),
]:
    print(f'{name:<32} {tr:.3f}      {te:.3f}     {l:.3f}')
for name, (tr, te, ll) in results_fi.items():
    print(f'{name:<32} {tr:.3f}      {te:.3f}     {ll:.3f}')
for name, tr, te, l in [
    ('lr graded (earlier)',            0.718, 0.706, 0.606),
    ('lr behavior-only (earlier)',     0.714, 0.705, 0.632),
]:
    print(f'{name:<32} {tr:.3f}      {te:.3f}     {l:.3f}')

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-e1bfd78de1f14115b5e89d0b134c3641?o=7474655510040746


lightgbm_fully_independent run: 83caab0e0522432e9f8a5826af36d247


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-c3a852375ec34d8bbdc9f854ca10c251?o=7474655510040746


lr_fully_independent run: 552a00d7e4a748fc951cbfd5328a49e6

variant                        train_auc  test_auc  log_loss
lightgbm graded (earlier)        0.756      0.713     0.594
lightgbm behavior-only (earlier) 0.755      0.712     0.607
lightgbm_fully_independent       0.748      0.702     0.601
lr_fully_independent             0.702      0.689     0.618
lr graded (earlier)              0.718      0.706     0.606
lr behavior-only (earlier)       0.714      0.705     0.632


In [20]:
# Evaluate the fully independent LightGBM: how much separation survives on raw
# borrower behavior alone, and which behavioral attributes carry it.

import numpy as np
import pandas as pd

prob_fi = lgbm_fi.predict_proba(X_test_fi)[:, 1]

eval_df = pd.DataFrame({'prob': prob_fi, 'actual': y_test.values})
eval_df['decile'] = pd.qcut(eval_df['prob'].rank(method='first'), 10,
                            labels=[f'D{i}' for i in range(1, 11)])
decile_table = eval_df.groupby('decile', observed=True).agg(
    n=('actual', 'count'),
    avg_predicted=('prob', 'mean'),
    actual_rate=('actual', 'mean'),
)
decile_table['lift'] = decile_table['actual_rate'] / eval_df['actual'].mean()
print(decile_table.round(3))

gain = pd.Series(
    lgbm_fi.booster_.feature_importance(importance_type='gain'),
    index=X_train_fi.columns,
).sort_values(ascending=False)
print()
print('top 15 behavior features by split-gain share:')
print((gain / gain.sum()).head(15).round(3))

            n  avg_predicted  actual_rate   lift
decile                                          
D1      51796          0.133        0.047  0.210
D2      51796          0.231        0.092  0.410
D3      51796          0.297        0.129  0.573
D4      51795          0.353        0.160  0.714
D5      51796          0.405        0.192  0.854
D6      51796          0.456        0.224  1.000
D7      51795          0.510        0.263  1.173
D8      51796          0.572        0.303  1.351
D9      51796          0.648        0.358  1.595
D10     51796          0.765        0.475  2.120

top 15 behavior features by split-gain share:
term                     0.306
fico_range_low           0.162
dti                      0.077
annual_inc               0.057
revol_bal                0.046
loan_amnt                0.045
tot_cur_bal              0.044
inq_last_6mths           0.033
credit_history_months    0.031
mort_acc                 0.028
revol_util               0.023
emp_length              

In [21]:
# Round 2 feature engineering: classic affordability ratios (row-wise arithmetic on
# columns already present, so nothing is learned from the data -> no leakage risk)
# plus a cap on the junk credit-history outliers (999-month entries from bad
# earliest_cr_line years).

X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

for X_ in (X_train_fe, X_test_fe):
    monthly_inc = X_['annual_inc'].clip(lower=1) / 12
    X_['loan_to_income'] = X_['loan_amnt'] / X_['annual_inc'].clip(lower=1)
    X_['payment_to_income'] = X_['installment'] / monthly_inc
    X_['revol_to_income'] = X_['revol_bal'] / X_['annual_inc'].clip(lower=1)
    X_['mortgage_share'] = X_['mort_acc'] / X_['total_acc'].clip(lower=1)
    X_['credit_history_months'] = X_['credit_history_months'].clip(upper=720)

print(f'features: {X_train.shape[1]} -> {X_train_fe.shape[1]}')
print()
print(X_train_fe[['loan_to_income', 'payment_to_income', 'revol_to_income', 'mortgage_share']].describe().round(3))

features: 52 -> 56

       loan_to_income  payment_to_income  revol_to_income  mortgage_share
count      778940.000         778940.000       778940.000      778940.000
mean            0.219              0.080            0.236           0.067
std             0.111              0.041            0.206           0.078
min             0.000              0.000            0.000           0.000
25%             0.133              0.049            0.108           0.000
50%             0.206              0.075            0.192           0.045
75%             0.298              0.107            0.310           0.111
max            10.000              4.459           16.401           0.769


In [22]:
# Train LightGBM on the enriched feature set — same fixed hyperparameters as every
# other LGBM run, so the comparison is purely about the features. Variant tag
# 'enriched_ratios' for filtering in the MLflow experiment.

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')

lgbm_fe = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=100, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1,
)

with mlflow.start_run(run_name='lightgbm_enriched_ratios') as run:
    lgbm_fe.fit(X_train_fe, y_train)
    auc_tr = roc_auc_score(y_train, lgbm_fe.predict_proba(X_train_fe)[:, 1])
    prob_test = lgbm_fe.predict_proba(X_test_fe)[:, 1]
    auc_te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'enriched_ratios')
    mlflow.log_params({'model': 'lightgbm', 'n_estimators': 500, 'learning_rate': 0.05,
                       'num_leaves': 63, 'n_features': X_train_fe.shape[1],
                       'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': auc_tr, 'test_auc': auc_te, 'test_log_loss': ll})
    signature = infer_signature(X_train_fe.head(100), lgbm_fe.predict_proba(X_train_fe.head(100))[:, 1])
    mlflow.sklearn.log_model(lgbm_fe, name='model', signature=signature, input_example=X_train_fe.head(3))
    print(f'lightgbm_enriched_ratios run: {run.info.run_id}')

print()
print('variant                        train_auc  test_auc  log_loss')
for name, tr, te, l in [
    ('lightgbm graded (base)',     0.756, 0.713, 0.594),
    ('lightgbm behavior-only',    0.755, 0.712, 0.607),
    ('lightgbm fully independent', 0.746, 0.701, 0.603),
]:
    print(f'{name:<32} {tr:.3f}      {te:.3f}     {l:.3f}')
print(f'{"lightgbm enriched_ratios":<32} {auc_tr:.3f}      {auc_te:.3f}     {ll:.3f}')

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-241589d39d1d444bb7a1778338c607c5?o=7474655510040746


lightgbm_enriched_ratios run: 7e8b6dd935f3460e87955ba76dfb15ef

variant                        train_auc  test_auc  log_loss
lightgbm graded (base)           0.756      0.713     0.594
lightgbm behavior-only           0.755      0.712     0.607
lightgbm fully independent       0.746      0.701     0.603
lightgbm enriched_ratios         0.759      0.714     0.592


In [23]:
# Did the new ratios earn their place? Check where they rank in split gain,
# plus a compact ranking summary.

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, brier_score_loss

prob_fe = lgbm_fe.predict_proba(X_test_fe)[:, 1]
fpr, tpr, _ = roc_curve(y_test, prob_fe)
print(f'KS: {np.max(tpr - fpr):.3f}   Brier: {brier_score_loss(y_test, prob_fe):.3f}')

gain = pd.Series(
    lgbm_fe.booster_.feature_importance(importance_type='gain'),
    index=X_train_fe.columns,
).sort_values(ascending=False)
new_ratios = ['loan_to_income', 'payment_to_income', 'revol_to_income', 'mortgage_share']
top = (gain / gain.sum()).head(15).round(3)
print()
print('top 15 features by split-gain share (new ratios marked *):')
print(top.rename(lambda c: f'{c} *' if c in new_ratios else c))

KS: 0.310   Brier: 0.204

top 15 features by split-gain share (new ratios marked *):
sub_grade                0.341
grade                    0.176
term                     0.086
dti                      0.039
tot_cur_bal              0.028
fico_range_low           0.026
mortgage_share *         0.023
int_rate                 0.022
loan_to_income *         0.021
credit_history_months    0.021
annual_inc               0.021
emp_length               0.018
payment_to_income *      0.018
revol_bal                0.017
revol_to_income *        0.016
dtype: float64


## 5. Hyperparameter tuning (temporal validation)

The test set stays untouched: the search fits on 2012–2014 and early-stops on a 2015
validation tail carved from the TRAIN window only. 25 configurations are sampled with a
fixed seed (trial 0 = the fixed challenger settings as a reference point). The winner
is refit on the full train window and evaluated ONCE on the test set in the next
section.

In [24]:
# HYPERPARAMETER TUNING — setup.
# The test set (2016-2018) stays UNTOUCHED until the final evaluation: all search and
# early stopping uses a temporal validation split carved from the TRAIN window only —
# fit on 2012-2014, validate on 2015 (the most recent train vintages). This mirrors how
# the model will be used in production: learned from the past, scored on the future.

import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
from sklearn.metrics import roc_auc_score, log_loss

# Re-derive issue months for the training rows. df_lending_club still carries issue_d
# and its row order matches X_train (built from the same temporal mask with reset_index).
issue_month = pd.to_datetime(df_lending_club['issue_d'], format='%b-%Y')
train_mask = issue_month < pd.Timestamp('2016-01-01')
train_months = issue_month[train_mask].reset_index(drop=True)
assert len(train_months) == len(X_train), 'row alignment between issue_d and X_train broken'

val_mask = train_months >= pd.Timestamp('2015-01-01')
X_tr, y_tr = X_train[~val_mask], y_train[~val_mask]
X_val, y_val = X_train[val_mask], y_train[val_mask]

print(f'fit:      {X_tr.shape[0]:,} rows (2012-2014), default rate {y_tr.mean():.1%}')
print(f'validate: {X_val.shape[0]:,} rows (2015),      default rate {y_val.mean():.1%}')

# Search space. Trial 0 is the current fixed challenger configuration (leaves 63,
# lr 0.05, min_child 100 — the 0.714 test-AUC run) so every search result is directly
# comparable to it. Trials 1-24 are sampled from the space with a fixed seed.
SEARCH_SPACE = {
    'num_leaves':        [31, 63, 95, 127],
    'learning_rate':    [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100, 200],
    'colsample_bytree':  [0.6, 0.8, 1.0],
    'subsample':         [0.7, 0.85, 1.0],
    'reg_alpha':         [0.0, 0.1, 1.0],
    'reg_lambda':        [0.0, 1.0, 5.0],
}

CONFIGS = [{
    'num_leaves': 63, 'learning_rate': 0.05, 'min_child_samples': 100,
    'colsample_bytree': 1.0, 'subsample': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
}]
seen = {tuple(sorted(CONFIGS[0].items()))}
rng = np.random.RandomState(42)
while len(CONFIGS) < 25:
    cfg = {k: v[rng.randint(len(v))] for k, v in SEARCH_SPACE.items()}
    key = tuple(sorted(cfg.items()))
    if key not in seen:
        seen.add(key)
        CONFIGS.append(cfg)

def evaluate_config(cfg, trial_idx):
    """Fit one config with early stopping on the 2015 validation tail (log-loss),
    record val AUC / log-loss / best iteration, and log the trial to MLflow."""
    model = lgb.LGBMClassifier(
        objective='binary', n_estimators=1000, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1, **cfg)
    t0 = time.time()
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    prob_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, prob_val)
    ll = log_loss(y_val, prob_val)
    with mlflow.start_run(run_name=f'lgbm_tuning_{trial_idx:02d}'):
        mlflow.set_tag('variant', 'tuning')
        mlflow.log_params({**cfg, 'n_estimators_max': 1000, 'early_stopping_rounds': 100,
                           'validation': '2015_train_tail'})
        mlflow.log_metrics({'val_auc': auc, 'val_log_loss': ll,
                            'best_iteration': model.best_iteration_,
                            'fit_seconds': round(time.time() - t0, 1)})
    return {'trial': trial_idx, **cfg, 'best_iter': model.best_iteration_,
            'val_auc': auc, 'val_log_loss': ll, 'seconds': round(time.time() - t0, 1)}

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')
tuning_results = []
print(f'{len(CONFIGS)} configs ready (trial 0 = current challenger settings)')

fit:      403,551 rows (2012-2014), default rate 17.2%
validate: 375,389 rows (2015),      default rate 20.2%
25 configs ready (trial 0 = current challenger settings)


In [25]:
# Random search — all 25 trials (trial 0 = the current challenger configuration as a
# reference point). Each trial takes ~80s on this data; the full sweep runs ~35 minutes
# and logs one MLflow run per trial (variant='tuning'). The five original execution
# chunks were merged into this single cell; the ranking cell below shows the completed
# 25-trial results, and all 25 runs remain in MLflow.

for i in range(len(CONFIGS)):
    res = evaluate_config(CONFIGS[i], i)
    tuning_results.append(res)
    print(res)

print(f'\ntotal trials completed: {len(tuning_results)}')

In [26]:
# Rank all trials by validation AUC and identify the winner.

tuning_df = pd.DataFrame(tuning_results).sort_values('val_auc', ascending=False).reset_index(drop=True)
print(tuning_df.round(4).head(10).to_string(index=False))

best = tuning_df.iloc[0]
baseline_auc = tuning_df.loc[tuning_df['trial'] == 0, 'val_auc'].iloc[0]
print()
print(f'best trial: {int(best.trial)}, val AUC {best.val_auc:.4f} vs trial-0 baseline {baseline_auc:.4f}')
print(f'best iteration on validation: {int(best.best_iter)} trees')

 trial  num_leaves  learning_rate  min_child_samples  colsample_bytree  subsample  reg_alpha  reg_lambda  best_iter  val_auc  val_log_loss  seconds
    10          63           0.03                200               0.8       0.85        0.1         1.0        223   0.7360        0.5969     31.4
    11          63           0.05                200               0.8       0.70        1.0         1.0        133   0.7359        0.5962     23.2
    15          31           0.10                 20               0.6       0.70        1.0         0.0         71   0.7356        0.5955     13.8
     4          31           0.05                 50               0.6       0.70        0.0         5.0        125   0.7353        0.5942     18.3
     3          63           0.03                 50               0.8       0.85        0.1         0.0       1000   0.7350        0.5857     82.8
    21          95           0.03                 20               0.8       1.00        1.0         1.0       1

## 6. Final model

Refit the winning configuration on the full 2012–2015 window (the 2015 validation tail
is training data, not test data) with the tree count fixed at the early-stopping
iteration, then a single evaluation on the untouched 2016–2018 test window — the only
time the test set sees a tuned model.

In [27]:
# FINAL MODEL: refit the winning configuration on the FULL training window (2012-2015,
# including the 2015 validation tail — it is training data, not test data) with the
# tree count fixed at the early-stopping iteration, then evaluate ONCE on the untouched
# 2016-2018 test set. This is the only time the test set sees a tuned model.

import time
import numpy as np
import mlflow
import mlflow.sklearn
import lightgbm as lgb
from mlflow.models import infer_signature
from sklearn.metrics import roc_auc_score, log_loss, roc_curve, brier_score_loss

best_cfg = {k: best[k] for k in SEARCH_SPACE}
# cast integer-valued params back to int (the results DataFrame stores them as floats)
best_cfg['num_leaves'] = int(best_cfg['num_leaves'])
best_cfg['min_child_samples'] = int(best_cfg['min_child_samples'])
n_trees = int(best['best_iter'])

lgbm_tuned = lgb.LGBMClassifier(
    objective='binary', n_estimators=n_trees, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1, **best_cfg)

start = time.time()
with mlflow.start_run(run_name='lightgbm_tuned_final') as run:
    lgbm_tuned.fit(X_train, y_train)

    prob_train = lgbm_tuned.predict_proba(X_train)[:, 1]
    prob_test = lgbm_tuned.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, prob_train)
    test_auc = roc_auc_score(y_test, prob_test)
    test_logloss = log_loss(y_test, prob_test)
    fpr, tpr, _ = roc_curve(y_test, prob_test)
    ks = float(np.max(tpr - fpr))
    brier = brier_score_loss(y_test, prob_test)

    mlflow.set_tag('variant', 'tuned_final')
    mlflow.log_params({**best_cfg, 'n_estimators': n_trees, 'class_weight': 'balanced',
                       'n_features': X_train.shape[1], 'n_train': X_train.shape[0],
                       'n_test': X_test.shape[0], 'split': 'temporal_2016_01_01'})
    mlflow.log_metrics({'train_auc': train_auc, 'test_auc': test_auc,
                        'test_log_loss': test_logloss, 'test_ks': ks, 'test_brier': brier})
    signature = infer_signature(X_train.head(100), prob_train[:100])
    mlflow.sklearn.log_model(lgbm_tuned, name='model', signature=signature, input_example=X_train.head(3))
    print(f'MLflow run: {run.info.run_id}')

print(f'trained in {time.time() - start:.0f}s')
print(f'AUC — train: {train_auc:.3f}, test: {test_auc:.3f}   (fixed challenger: 0.714, LR: 0.707)')
print(f'test log-loss: {test_logloss:.3f}  (fixed challenger: 0.592, LR: 0.603)')
print(f'KS: {ks:.3f}   Brier: {brier:.3f}')
print(f'winning config: {best_cfg} | n_estimators: {n_trees}')

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-411fdf84c85d4f6cbb45d32d9cc7831e?o=7474655510040746


MLflow run: 489e950cec03422897d53aecbb74f746
trained in 51s
AUC — train: 0.734, test: 0.713   (fixed challenger: 0.714, LR: 0.707)
test log-loss: 0.602  (fixed challenger: 0.592, LR: 0.603)
KS: 0.310   Brier: 0.208
winning config: {'num_leaves': 63, 'learning_rate': np.float64(0.03), 'min_child_samples': 200, 'colsample_bytree': np.float64(0.8), 'subsample': np.float64(0.85), 'reg_alpha': np.float64(0.1), 'reg_lambda': np.float64(1.0)} | n_estimators: 223


## 7. Variant for the model-choice discussion

Drop `grade`, `sub_grade` AND `int_rate` while keeping `installment` — the variant used
in the business-impact analysis to quantify how much signal Lending Club’s grading and
pricing contribute. `installment` is mechanically derived from amount/term/rate by the
amortization formula, so it carries part of the pricing signal back; this variant sits
between behavior-only and fully independent.

In [28]:
# Variant for the model-choice discussion: remove Lending Club's grading AND its
# primary pricing lever — grade, sub_grade, int_rate — while KEEPING installment.
# Caveat: installment is mechanically derived from loan_amnt, term and int_rate by
# the amortization formula, so it still carries part of the pricing signal. This
# variant therefore sits between behavior-only (int_rate kept, AUC 0.713) and the
# fully independent variant (installment also removed, AUC 0.702).

import time
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

X_train_nogp = X_train.drop(columns=['grade', 'sub_grade', 'int_rate'])
X_test_nogp = X_test.drop(columns=['grade', 'sub_grade', 'int_rate'])
print(f'features: {X_train.shape[1]} -> {X_train_nogp.shape[1]} (dropped grade, sub_grade, int_rate)')

mlflow.set_experiment('/Users/<your_username>/credit_risk_lending_club')
results_nogp = {}

# --- LightGBM, same fixed challenger hyperparameters for a fair comparison ---
lgbm_nogp = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=100, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1,
)
with mlflow.start_run(run_name='lightgbm_no_grade_rate') as run:
    lgbm_nogp.fit(X_train_nogp, y_train)
    tr = roc_auc_score(y_train, lgbm_nogp.predict_proba(X_train_nogp)[:, 1])
    prob_test = lgbm_nogp.predict_proba(X_test_nogp)[:, 1]
    te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'no_grade_rate')
    mlflow.log_params({'model': 'lightgbm', 'n_estimators': 500, 'learning_rate': 0.05,
                       'num_leaves': 63, 'n_features': X_train_nogp.shape[1],
                       'split': 'temporal_2016_01_01', 'dropped': 'grade,sub_grade,int_rate'})
    mlflow.log_metrics({'train_auc': tr, 'test_auc': te, 'test_log_loss': ll})
    signature = infer_signature(X_train_nogp.head(100), lgbm_nogp.predict_proba(X_train_nogp.head(100))[:, 1])
    mlflow.sklearn.log_model(lgbm_nogp, name='model', signature=signature, input_example=X_train_nogp.head(3))
    print(f'lightgbm_no_grade_rate run: {run.info.run_id}')
results_nogp['lightgbm_no_grade_rate'] = (tr, te, ll)

# --- Logistic regression, same setup as the graded baseline ---
lr_nogp = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
with mlflow.start_run(run_name='lr_no_grade_rate') as run:
    lr_nogp.fit(X_train_nogp, y_train)
    tr = roc_auc_score(y_train, lr_nogp.predict_proba(X_train_nogp)[:, 1])
    prob_test = lr_nogp.predict_proba(X_test_nogp)[:, 1]
    te = roc_auc_score(y_test, prob_test)
    ll = log_loss(y_test, prob_test)
    mlflow.set_tag('variant', 'no_grade_rate')
    mlflow.log_params({'model': 'logistic_regression', 'n_features': X_train_nogp.shape[1],
                       'split': 'temporal_2016_01_01', 'dropped': 'grade,sub_grade,int_rate'})
    mlflow.log_metrics({'train_auc': tr, 'test_auc': te, 'test_log_loss': ll})
    signature = infer_signature(X_train_nogp.head(100), lr_nogp.predict_proba(X_train_nogp.head(100))[:, 1])
    mlflow.sklearn.log_model(lr_nogp, name='model', signature=signature, input_example=X_train_nogp.head(3))
    print(f'lr_no_grade_rate run: {run.info.run_id}')
results_nogp['lr_no_grade_rate'] = (tr, te, ll)

print()
print('variant                                     train_auc  test_auc  log_loss')
for name, tr, te, l in [
    ('lightgbm graded (all features)',       0.758, 0.714, 0.592),
    ('lightgbm behavior-only (-grade)',       0.756, 0.713, 0.605),
    ('lightgbm fully independent (-inst)',   0.748, 0.702, 0.601),
]:
    print(f'{name:<42} {tr:.3f}      {te:.3f}     {l:.3f}')
for name, (tr, te, ll) in results_nogp.items():
    print(f'{name:<42} {tr:.3f}      {te:.3f}     {ll:.3f}')
for name, tr, te, l in [
    ('lr graded (all features)',              0.718, 0.707, 0.603),
    ('lr fully independent (-inst)',         0.702, 0.689, 0.618),
]:
    print(f'{name:<42} {tr:.3f}      {te:.3f}     {l:.3f}')

# What carries the signal once LC grading and pricing are gone?
import pandas as pd
gain = pd.Series(
    lgbm_nogp.booster_.feature_importance(importance_type='gain'),
    index=X_train_nogp.columns,
).sort_values(ascending=False)
print()
print('top 12 features by split-gain share (no grade / sub_grade / int_rate):')
print((gain / gain.sum()).head(12).round(3))

features: 52 -> 49 (dropped grade, sub_grade, int_rate)


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-1f216745ba774611826f630b2a634614?o=7474655510040746


lightgbm_no_grade_rate run: 34bac982fe954f01bc35d5c364df0cfd


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-64cccd07-53bd.cloud.databricks.com/ml/experiments/2378456567278483/models/m-10c05c62ced24fa5b4caaeeb6a529932?o=7474655510040746


lr_no_grade_rate run: 8214d86a5ae54f0e8e35a4fe11283ea7

variant                                     train_auc  test_auc  log_loss
lightgbm graded (all features)             0.758      0.714     0.592
lightgbm behavior-only (-grade)            0.756      0.713     0.605
lightgbm fully independent (-inst)         0.748      0.702     0.601
lightgbm_no_grade_rate                     0.753      0.710     0.603
lr_no_grade_rate                           0.707      0.698     0.622
lr graded (all features)                   0.718      0.707     0.603
lr fully independent (-inst)               0.702      0.689     0.618

top 12 features by split-gain share (no grade / sub_grade / int_rate):
term                     0.296
fico_range_low           0.155
installment              0.073
dti                      0.073
annual_inc               0.055
revol_bal                0.044
tot_cur_bal              0.040
inq_last_6mths           0.030
loan_amnt                0.028
credit_history_months    0.02

## Summary of modeling results

All metrics are out-of-time test performance (2016–2018 window) produced by the executed
cells above; MLflow run IDs link each number to its tracked artifact.

| Variant | Test AUC | Test log-loss | MLflow run |
| --- | --- | --- | --- |
| Logistic regression — 52 features | 0.707 | 0.603 | `0ee58eab` |
| **LightGBM fixed challenger — 52 features (recommended)** | **0.714** | **0.592** | `018dcac1` |
| LightGBM tuned — random-search winner | 0.713 | 0.602 | `489e950c` |
| LightGBM — grading removed (`grade`, `sub_grade`) | 0.713 | 0.605 | — |
| LightGBM — grading + rate removed | 0.710 | 0.603 | `34bac982` |
| LightGBM — fully independent (no LC grading/pricing) | 0.702 | 0.601 | `83caab0e` |

**Findings**

* Most of the signal is Lending Club’s own grading/pricing (`sub_grade` 34% + `grade` 19% of split gain), and it is nearly linear — trees add little over logistic regression (0.714 vs 0.707 AUC).
* Raw borrower behavior alone retains most of the ranking power (AUC 0.702 of a possible 0.714): `term`, FICO and DTI carry it.
* Affordability ratios add nothing for LightGBM (0.714 → 0.714) — trees already compose ratios internally; feature engineering must add *new information*, not re-express existing columns.
* Tuning halves the overfit gap (train−test AUC 0.021 vs 0.044) but does not improve out-of-time ranking — the signal plateau is real, not a tuning artifact.

**Recommendation (carried into the business-impact notebook):** deploy the fixed
LightGBM challenger with isotonic calibration as the PD scorer; the
`04_business_impact` notebook converts these scores into lending economics and the
approval-policy recommendation.